In [1]:
# две самые главные вещи
from revpy import reversible   # декоратор, генерирующий код для обратной функции
from revpy import ReversibleFunction   # удобный класс для объединения двух обратных функций в одну обратимую

# синтаксические конструкции, чтобы обычный python не ругался
from revpy import fi, elihw

# опционально: для проверки обратимости в runtime и compile-time
from revpy import reversibility_assert, ReversibilityError, RevpyCompilationError

# Базовые обратимые функции



Обратимые функции revpy состоят в первую очередь из вызовов других обратимых функций.

Откуда же эти обратимые функции взять изначально? -- их нужно написать руками, самостоятельно.

Это очень просто!
Ниже показано, как это сделать.



In [2]:

# руками пишем две функции, и постулируем что они обратны друг другу
# в данном случае: вычитание константы и добавление константы
add = ReversibleFunction(
    f=lambda x, a: x + a, 
    f_rev=lambda x, a: x - a
)

# после постулирования можно использовать такой синтаксис
# для вызова прямой функции
assert add(1)[2] == 3
assert add(a=1)[2] == 3

# для вызова обратной функции
assert add(1).rev[2] == 1
assert add.rev(a=1)[2] == 1

# умножение на константу и деление на константу
mult = ReversibleFunction(
    f=lambda x, a: x * a, 
    f_rev=lambda x, a: x / a
)

# функция которая ничего не делает (название move взято от move-семантики из C++)
move = ReversibleFunction(
    f=lambda x: x,
    f_rev=lambda x: x,
)


# функция инициализации переменной константным значением
# обратная функция - уничтожение этой переменной с утверждением о том, что она равна этой константе

def init(none, value, compare=lambda x, y: x == y):
    reversibility_assert(none is None, none)
    # assert none is None, none    # это то же самое, что строка выше, просто будет кидаться AssertionError вместо ReversibilityError
    return value

def destroy(x, value, compare=lambda x, y: x == y):
    reversibility_assert(compare(x, value), (x, value))
    # assert compare(x, value), (x, value)
    return None

# инициализация и уничтожение для обычных объектов
init = ReversibleFunction(init, destroy)
destroy = init.rev

# инициализация и уничтожение для float, где обратимость имеется с точностью до погрешности вычислений
init_float = init(compare=lambda x, y: abs(x - y) < 1e-5)
destroy_float = init_float.rev

# Написание обратимых функций

Ниже показаны примеры, как с помощью декоратора `@reversible` можно очень удобно писать обратимые программы. Как будто на настоящем обратимом языке программирования, но только это python.

Каждая строка должна иметь вид:

`создаваемые переменные = <выражение, возвращающее обратимую функцию>[уничтожаемые переменные]`

- обратимой функцией считается всё, у чего есть `__getitem__` (оператор вызова обратимой функции) и `.rev` (оператор взятия обратной функции)


Время жизни всех перменных проверяется статически, поэтому о многих ошибках вы узнаете до вызова функции.

In [3]:
@reversible
def affine(
    x,      # первый аргумент считается входом
    a, b    # остальные считаются параметрами обратимого преобразования
):
    y = mult(a)[x]
    z = add(b)[y]
    return z

print(affine(a=2, b=3)[1.0])   # получится 5.0
print(affine(2, 3).rev[5.0])   # получится 1.0

5.0
1.0


In [4]:
@reversible
def test_if(x, a, b, condition: bool):
    if condition:
        x = mult(a)[x]
        y = add(b)[x]
        
    else:
        y = move[x]

    fi(condition)   # опционально
    return y


print(test_if(3, 2, True)[10])
print(test_if(3, 2, True).rev[10])
print(test_if(3, 2, False).rev[10])

32
2.6666666666666665
10


In [5]:
@reversible
def test_while(x, a, N=10):
    n = init(0)[None]
    while n < N:
        x = add(a)[x]
        n = add(1)[n]
    elihw(n > 0)
    destroy(N)[n]
    return x

print(test_while(a=10, N=10)[7])  # 107
print(test_while(a=10, N=10).rev[107])  # 7

107
7


# Что же происходит под капотом?

Если есть вопросы, почему что-то не работает -- взгляните на явно-сгенерированный код прямой и обратной функций!

Он доступен через `reversible_function.src` и `reversible_function.rev.src`. Это буквально тот код, который в итоге запускается.

In [6]:
print(test_while.src)

def test_while(x, a, N=10):
    n = init(0)[None]
    reversibility_assert(not n > 0, "reversibility_assert is False: 'not n > 0'")
    while n < N:
        x = add(a)[x]
        n = add(1)[n]
        reversibility_assert(n > 0, "reversibility_assert is False: 'n > 0'")
    reversibility_assert(destroy(N)[n] is None, "reversibility_assert is False: 'destroy(N)[n] is None'")
    return x


In [7]:
print(test_while.rev.src)

def test_while_rev(x, a, N=10):
    n = destroy(N).rev[None]
    reversibility_assert(not n < N, "reversibility_assert is False: 'not n < N'")
    while n > 0:
        n = add(1).rev[n]
        x = add(a).rev[x]
        reversibility_assert(n < N, "reversibility_assert is False: 'n < N'")
    reversibility_assert(init(0).rev[n] is None, "reversibility_assert is False: 'init(0).rev[n] is None'")
    return x


# Примеры ошибок

In [8]:
def _affine(x, a, b):
    y = mult(a)[x]
    # тут переменная x уничтожена ([переменная] обозначает уничтожение переменной вызовом обратимой функции)
    z = add(b)[x]  # ошибка компиляции: использование уничтоженной переменной
    return z

try:
    _affine = reversible(_affine)
    assert False
except RevpyCompilationError as e:
    print(e)

Lifetime error: variable 'x'
	--> y = mult(a)[x]
	--> z = add(b)[x] 


In [9]:
@reversible
def _test_while(x, a, N=10):
    n = init(0)[None]
    while n < N:
        x = add(a)[x]
        n = add(1)[n]
    elihw(n > 1)
    destroy(N)[n]
    return x

try:
    _test_while(a=10, N=10)[7]
    _test_while(a=10, N=10).rev[107]
except ReversibilityError as e:
    print(e)

reversibility_assert is False: 'n > 1'


# Print in code

In [10]:
@reversible
def test_while(x, a, N=10):
    n = init(0)[None]
    while n < N:
        x = add(a)[x]
        n = add(1)[n]
        print(x, end=' ')
    elihw(n > 0)
    destroy(N)[n]
    return x

test_while(a=10, N=10)[7]
print()
test_while(a=10, N=10).rev[107]

17 27 37 47 57 67 77 87 97 107 
107 97 87 77 67 57 47 37 27 17 

7

In [11]:
print(test_while.src)
print('==========')
print(test_while.rev.src)

def test_while(x, a, N=10):
    n = init(0)[None]
    reversibility_assert(not n > 0, "reversibility_assert is False: 'not n > 0'")
    while n < N:
        x = add(a)[x]
        n = add(1)[n]
        print(x, end=' ')
        reversibility_assert(n > 0, "reversibility_assert is False: 'n > 0'")
    reversibility_assert(destroy(N)[n] is None, "reversibility_assert is False: 'destroy(N)[n] is None'")
    return x
def test_while_rev(x, a, N=10):
    n = destroy(N).rev[None]
    reversibility_assert(not n < N, "reversibility_assert is False: 'not n < N'")
    while n > 0:
        print(x, end=' ')
        n = add(1).rev[n]
        x = add(a).rev[x]
        reversibility_assert(n < N, "reversibility_assert is False: 'n < N'")
    reversibility_assert(init(0).rev[n] is None, "reversibility_assert is False: 'init(0).rev[n] is None'")
    return x


# Числа Фибоначчи

In [12]:
from revpy import reversible, ReversibleFunction, elihw

# (a, b) ↔ (b, a+b)
fib_step = ReversibleFunction(
    f=lambda ab: (ab[1], ab[0] + ab[1]),
    f_rev=lambda cd: (cd[1] - cd[0], cd[0]),
)

In [13]:
@reversible
def fib(n):
    c = move[n]                       # счётчик = n, n уничтожается
    x1 = init(1)[None]                # x1 = 1
    x2 = init(1)[None]                # x2 = 1
    while c > 0:
        x1, x2 = fib_step[x1, x2]     # (a, b) → (b, a+b)
        c = add(-1)[c]                # c -= 1
    elihw(not (x1 == 1 and x2 == 1))  # обратное условие: дошли до базы
    destroy(0)[c]                     # c должен стать 0
    res = move[x1, x2]
    return res                     # (fib(n+1), fib(n+2))

In [14]:
# Прямой ход: n → (fib(n+1), fib(n+2))
assert fib()[0]  == (1, 1)         # fib(0) → (fib(1), fib(2))
assert fib()[1]  == (1, 2)
assert fib()[2]  == (2, 3)
assert fib()[5]  == (8, 13)
assert fib()[10] == (89, 144)

# Обратный ход: (fib(n+1), fib(n+2)) → n
assert fib().rev[1, 1]    == 0
assert fib().rev[1, 2]    == 1
assert fib().rev[8, 13]   == 5
assert fib().rev[89, 144] == 10

# Обратимый алгоритм Рабина-Карпа

In [15]:
add = ReversibleFunction(
    f=lambda x, a: x + a,
    f_rev=lambda x, a: x - a,
)

mod_add = ReversibleFunction(
    f=lambda x, a, q: (x + a) % q,
    f_rev=lambda y, a, q: (y - a) % q,
)

mod_sub = ReversibleFunction(
    f=lambda x, a, q: (x - a) % q,
    f_rev=lambda y, a, q: (y + a) % q,
)

mod_mul = ReversibleFunction(
    f=lambda x, a, q: (x * a) % q,
    f_rev=lambda y, a, q: (y * pow(a, -1, q)) % q,
)

push = ReversibleFunction(
    f=lambda stack, x: stack + [x],
    f_rev=lambda new_stack, x: new_stack[:-1],   # x — параметр (сохранён), возвращаем старый стек
)

In [16]:
@reversible
def update_hash(t, Ts, Tsm, h, d, q):
    # t' = ((t - Ts*h) * d + Tsm) mod q
    Ts_h = init(Ts * h % q)[None]
    t = mod_sub(Ts_h, q)[t]
    t = mod_mul(d, q)[t]
    t = mod_add(Tsm, q)[t]
    destroy(Ts * h % q)[Ts_h]
    return t

In [17]:
print(update_hash.src)

def update_hash(t, Ts, Tsm, h, d, q):
    Ts_h = init(Ts * h % q)[None]
    t = mod_sub(Ts_h, q)[t]
    t = mod_mul(d, q)[t]
    t = mod_add(Tsm, q)[t]
    reversibility_assert(destroy(Ts * h % q)[Ts_h] is None, "reversibility_assert is False: 'destroy(Ts * h % q)[Ts_h] is None'")
    return t


In [18]:
print(update_hash.rev.src)

def update_hash_rev(t, Ts, Tsm, h, d, q):
    Ts_h = destroy(Ts * h % q).rev[None]
    t = mod_add(Tsm, q).rev[t]
    t = mod_mul(d, q).rev[t]
    t = mod_sub(Ts_h, q).rev[t]
    reversibility_assert(init(Ts * h % q).rev[Ts_h] is None, "reversibility_assert is False: 'init(Ts * h % q).rev[Ts_h] is None'")
    return t


In [19]:
def poly_hash(arr, d, q):
    h = 0
    for x in arr:
        h = (h * d + x) % q
    return h


@reversible
def rabinkarp(T, P, d, q):
    n = init(len(T))[None]
    m = init(len(P))[None]
    R = init([])[None]

    if n >= m:
        # h = d^(m-1) mod q
        h = init(1)[None]
        if m > 1:
            i = init(0)[None]
            while i < m - 1:
                h = mod_mul(d, q)[h]
                i = add(1)[i]
            elihw(i > 0)
            destroy(m - 1)[i]
        fi(m > 1)

        # p = hash(P)
        p = init(0)[None]
        i = init(0)[None]
        while i < m:
            p = mod_add(P[i], q)[p]
            if i < m - 1:
                p = mod_mul(d, q)[p]
            fi(i < m - 1)
            i = add(1)[i]
        elihw(i > 0)
        destroy(m)[i]

        # t = hash(T[0:m])
        t = init(0)[None]
        i = init(0)[None]
        while i < m:
            t = mod_add(T[i], q)[t]
            if i < m - 1:
                t = mod_mul(d, q)[t]
            fi(i < m - 1)
            i = add(1)[i]
        elihw(i > 0)
        destroy(m)[i]

        # сканирование
        s = init(0)[None]
        while s <= n - m:
            if p == t:
                if T[s:s+m] == P:
                    R = push(s)[R]
                fi(T[s:s+m] == P)
            fi(p == t)

            if s < n - m:
                t = update_hash(T[s], T[s+m], h, d, q)[t]
            fi(s < n - m)

            s = add(1)[s]
        elihw(s > 0)
        destroy(n - m + 1)[s]

        destroy(poly_hash(T[:m], d, q))[t]
        destroy(poly_hash(P, d, q))[p]
        destroy(pow(d, m - 1, q))[h]
    fi(n >= m)

    destroy(len(T))[n]
    destroy(len(P))[m]

    res = move[T, R]    # ← consumes T (first arg) and R, returns (T, R)
    return res

In [20]:
print(rabinkarp.src)

def rabinkarp(T, P, d, q):
    n = init(len(T))[None]
    m = init(len(P))[None]
    R = init([])[None]
    if n >= m:
        h = init(1)[None]
        if m > 1:
            i = init(0)[None]
            reversibility_assert(not i > 0, "reversibility_assert is False: 'not i > 0'")
            while i < m - 1:
                h = mod_mul(d, q)[h]
                i = add(1)[i]
                reversibility_assert(i > 0, "reversibility_assert is False: 'i > 0'")
            reversibility_assert(destroy(m - 1)[i] is None, "reversibility_assert is False: 'destroy(m - 1)[i] is None'")
            reversibility_assert(m > 1, "reversibility_assert is False: 'm > 1'")
        else:
            reversibility_assert(not m > 1, "reversibility_assert is False: 'not m > 1'")
        p = init(0)[None]
        i = init(0)[None]
        reversibility_assert(not i > 0, "reversibility_assert is False: 'not i > 0'")
        while i < m:
            p = mod_add(P[i], q)[p]
            if i < m - 1:
      

In [21]:
print(rabinkarp.rev.src)

def rabinkarp_rev(res, P, d, q):
    T, R = move.rev[res]
    m = destroy(len(P)).rev[None]
    n = destroy(len(T)).rev[None]
    if n >= m:
        h = destroy(pow(d, m - 1, q)).rev[None]
        p = destroy(poly_hash(P, d, q)).rev[None]
        t = destroy(poly_hash(T[:m], d, q)).rev[None]
        s = destroy(n - m + 1).rev[None]
        reversibility_assert(not s <= n - m, "reversibility_assert is False: 'not s <= n - m'")
        while s > 0:
            s = add(1).rev[s]
            if s < n - m:
                t = update_hash(T[s], T[s + m], h, d, q).rev[t]
                reversibility_assert(s < n - m, "reversibility_assert is False: 's < n - m'")
            else:
                reversibility_assert(not s < n - m, "reversibility_assert is False: 'not s < n - m'")
            if p == t:
                if T[s:s + m] == P:
                    R = push(s).rev[R]
                    reversibility_assert(T[s:s + m] == P, "reversibility_assert is False: 'T[s:s + m] == P'")
       

In [22]:
T = [1, 2, 3, 1, 2, 3]
P = [1, 2, 3]
d, q = 10, 101

# Прямой проход: T разрушается, возвращается кортеж (T, R)
res = rabinkarp(P=P, d=d, q=q)[T]
T_out, R = res
print(R)          # [0, 3]
print(T_out)      # [1, 2, 3, 1, 2, 3]  (тот же T, прошёл через move)

# Обратный проход: кортеж (T, R) → восстановленный T
T_back = rabinkarp(P=P, d=d, q=q).rev[res]
print(T_back)     # [1, 2, 3, 1, 2, 3]

[0, 3]
[1, 2, 3, 1, 2, 3]
[1, 2, 3, 1, 2, 3]
